In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv("Team_Stats_Merge_2018_2025.csv")

## Variable Selection

* Using the information we discovered from the feature_selection_analysis file on which variables to remove.

In [3]:
key_cols = ['season', 'team']
target_col = 'made_playoffs'

In [4]:
drop_leakage = [
    "tp_W",       # Regular-season wins
    "tp_L",       # Regular-season losses
    "tp_W-L%",    # Win percentage
    "tp_PF",      # Points For
    "tp_PA",      # Points Against
    "tp_PD",      # Point Differential
    "tp_MoV",     # Margin of Victory
]

In [5]:
drop_srs = [
    "tp_SRS",     # Simple Rating System (MoV + SoS)
    "tp_OSRS",    # Offensive SRS component
    "tp_DSRS",    # Defensive SRS component
]

In [6]:
drop_games = [
    "off_g", "def_g", "off_pass_G", "off_rush_G",
    "def_pass_G", "def_rush_G", "def_adv_G",
]

In [7]:
drop_situational = [
    "off_pass_4QC",   # Fourth-quarter comebacks
    "off_pass_GWD",   # Game-winning drives
]

In [8]:
drop_low_variance = [
    "tp_T",           # Ties (near-zero variance)
]

In [9]:
drop_multicollinear_off_pass = [
    "off_pass_Y/A", "off_pass_AY/A", "off_pass_NY/A", "off_pass_EXP",
    "off_pass_Yds", "off_pass_Att", "off_pass_Cmp", "off_pass_TD",
]

In [10]:
drop_multicollinear_def_pass = [
    "off_pass_Lng", "off_pass_Y/C", "off_pass_Y/G", "off_pass_Yds.1",
    "def_pass_PD", "def_pass_Att", "def_pass_Cmp", "def_pass_Yds",
    "def_pass_TD", "def_pass_Y/A", "def_pass_AY/A", "def_pass_Y/C",
    "def_pass_Y/G", "def_pass_NY/A", "def_pass_Yds.1", "def_pass_EXP",
]

In [11]:
drop_rush_volume = [
    "off_rush_Lng", "off_rush_Y/G", "off_rush_Fmb", "def_rush_Y/G",
]

In [12]:
drop_adv_duplicates = [
    "def_adv_Sk",      # Identical to def_pass_Sk
    "def_adv_Att",     # Identical to def_pass_Att
    "def_adv_Cmp",     # Identical to def_pass_Cmp
    "def_adv_Yds",     # Identical to def_pass_Yds
    "def_adv_TD",      # Identical to def_pass_TD
]

In [13]:
drop_adv_null = [
    "def_adv_Bltz%",   # Blitz percentage
    "def_adv_Hrry%",   # Hurry percentage
    "def_adv_QBKD%",   # QB knockdown percentage
    "def_adv_Prss%",   # Pressure percentage
]

In [14]:
drop_adv_weak = [
    "def_adv_QBKD",    # QB knockdowns
    "def_adv_Hrry",    # Hurries
    "def_adv_Bltz",    # Blitzes
    "def_adv_DADOT",   # Depth of target
]

In [15]:
all_drops = (
    drop_leakage + drop_srs + drop_games + drop_situational +
    drop_low_variance + drop_multicollinear_off_pass +
    drop_multicollinear_def_pass + drop_rush_volume +
    drop_adv_duplicates + drop_adv_null + drop_adv_weak
)

In [17]:
# Verify no accidental duplicates in drop list
len(all_drops) == len(set(all_drops))

True

In [27]:
# Verify all columns exist in the raw data
missing_from_raw = [c for c in all_drops if c not in df_raw.columns]
if missing_from_raw:
    print(f"WARNING: These columns not found in raw data: {missing_from_raw}")

In [28]:
# Apply variable selection
df = df_raw.drop(columns=[c for c in all_drops if c in df_raw.columns]).copy()

In [29]:
dropped_count = len([c for c in all_drops if c in df_raw.columns])
print(f"Drop Count: {dropped_count}")
print(f"Dataset after selection: {df.shape}")
print(f"Feature columns:{df.shape[1] - len(key_cols) - 1}")

Drop Count: 61
Dataset after selection: (256, 60)
Feature columns:57


In [30]:
# List retained features
retained_features = [c for c in df.columns if c not in key_cols + [target_col]]
pd.DataFrame({"Retained Features": retained_features}, index=range(1, len(retained_features)+1))

,Retained Features
1,tp_SoS
2,off_pts
3,off_yds
4,off_plays
5,off_yds_per_play
6,off_turnovers
7,off_fumbles
8,off_penalties
9,off_penalty_yds
10,off_first_downs_by_penalty


In [31]:
# Assess missing values in retained dataset
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    display(pd.DataFrame([{"Status": "Dataset complete"}]))
else:
    missing_rows = []
    for col, count in missing.items():
        pct = count / len(df) * 100
        missing_rows.append({"Column": col, "Missing": count, "Missing %": f"{pct:.1f}%"})
    display(pd.DataFrame(missing_rows).set_index("Column"))


,Missing,Missing %
Column,,
def_pass_Int,1,0.4%
def_pass_Int%,1,0.4%


In [32]:
if "def_pass_Int" in df.columns and df["def_pass_Int"].isna().sum() > 0:
    missing_records = df[df["def_pass_Int"].isna()][["season", "team"]]
    display(missing_records)
 
# Impute using 2025 season median
if "def_pass_Int" in df.columns and df["def_pass_Int"].isna().sum() > 0:
    season_2025 = df[df["season"] == 2025]
    median_int = season_2025["def_pass_Int"].median()
    median_int_pct = season_2025["def_pass_Int%"].median()
 
    imputation_info = pd.DataFrame([
        {"Column": "def_pass_Int", "Imputed Value": median_int, "Method": "2025 season median"},
        {"Column": "def_pass_Int%", "Imputed Value": median_int_pct, "Method": "2025 season median"},
    ]).set_index("Column")
    display(imputation_info)
 
    df["def_pass_Int"] = df["def_pass_Int"].fillna(median_int)
    df["def_pass_Int%"] = df["def_pass_Int%"].fillna(median_int_pct)
 

,season,team
227,2025,New York Jets


,Imputed Value,Method
Column,,
def_pass_Int,11.0,2025 season median
def_pass_Int%,2.2,2025 season median


In [33]:
# Validating whether we have missings values still, or if we are now finished.
remaining_missing = df.isnull().sum().sum()
print(remaining_missing)
print(df.shape)

0
(256, 60)


Now, to begin identifying outliers we will begin scanning the retained features for values beyond +/- 3 standard deviations.

In [35]:
# Scanning for outliers 
numeric_features = [c for c in retained_features if df[c].dtype in ["float64", "int64"]]
outlier_rows = []

for col in numeric_features:
    mean = df[col].mean()
    std = df[col].std()
    if std == 0:
        continue
    outliers = df[(df[col] - mean).abs() > 3 * std]
    if len(outliers) > 0:
        for _, row in outliers.iterrows():
            z = (row[col] - mean) / std
            outlier_rows.append({
                "Feature": col,
                "Season": row["season"],
                "Team": row["team"],
                "Value": round(row[col], 2),
                "Z-Score": round(z, 2),
                "Mean": round(mean, 2),
                "Std": round(std, 2),
            })

outlier_df = pd.DataFrame(outlier_rows)
display(outlier_df)


,Feature,Season,Team,Value,Z-Score,Mean,Std
0,tp_SoS,2025,New England Patriots,-4.50,-3.84,0.00,1.17
1,off_yds,2018,Arizona Cardinals,3865.00,-3.04,5684.12,599.21
2,off_plays,2018,Miami Dolphins,878.00,-3.19,1045.86,52.64
3,off_plays,2019,Washington Commanders,885.00,-3.06,1045.86,52.64
4,off_turnovers,2019,Tampa Bay Buccaneers,41.00,3.54,21.52,5.50
5,off_fumbles,2023,New York Jets,18.00,3.20,8.71,2.91
6,off_turnover_pct,2019,Tampa Bay Buccaneers,20.20,3.18,11.29,2.80
7,def_plays,2021,Seattle Seahawks,1201.00,3.33,1045.86,46.55
8,def_turnovers_forced,2025,New York Jets,4.00,-3.09,21.52,5.67
9,def_fumbles_forced,2019,Pittsburgh Steelers,18.00,3.16,8.71,2.94


What these results reveal are the outlying stats from a team during a certain season. 

We can observe some results such as the off_rush_TD feature for the 2024 Buffalo Bills at 32 Touchdowns on the season with a z-score = +3.11 and the league mean/average of only 15.24 rushing Touchdowns that season. 

In [36]:
# Validation values 

dupes = df.duplicated(subset=["season", "team"]).sum()
print(dupes)

teams_per_season = df.groupby("season")["team"].nunique()
print(teams_per_season)

unique_teams = df["team"].nunique()
print(unique_teams)

print(df.dtypes.value_counts())


0
season
2018    32
2019    32
2020    32
2021    32
2022    32
2023    32
2024    32
2025    32
Name: team, dtype: int64
32
float64    31
int64      28
object      1
Name: count, dtype: int64


**Differential Features**

Here we are looking at the retained features, and chose pairs where one measures the offensive and defensive version of something in whcih we will then subtract them to create a 'competetive advantage' metric.

In [45]:
# Derived Differential Features
df["score_pct_diff"] = df["off_score_pct"] - df["def_score_pct"]
df["pass_eff_diff"] = df["off_pass_ANY/A"] - df["def_pass_ANY/A"]
df["ypp_diff"] = df["off_yds_per_play"] - df["def_yds_per_play"]
df["turnover_margin"] = df["def_turnovers_forced"] - df["off_turnovers"]
df["to_rate_diff"] = df["def_turnover_pct"] - df["off_turnover_pct"]
df["sack_rate_diff"] = df["def_pass_Sk%"] - df["off_pass_Sk%"]
df["rush_eff_diff"] = df["off_rush_Y/A"] - df["def_rush_Y/A"]

engineered_features = [
    "score_pct_diff", "pass_eff_diff", "ypp_diff",
    "turnover_margin", "to_rate_diff", "sack_rate_diff", "rush_eff_diff"
]

# Engineered feature correlations with made_playoffs
eng_corr_rows = []
for feat in engineered_features:
    corr = df[feat].corr(df["made_playoffs"])
    eng_corr_rows.append({"Feature": feat, "Correlation": round(corr, 3)})

display(pd.DataFrame(eng_corr_rows).set_index("Feature"))

# Raw component correlations for comparison
raw_comparisons = [
    ("off_score_pct", "def_score_pct"),
    ("off_pass_ANY/A", "def_pass_ANY/A"),
    ("off_yds_per_play", "def_yds_per_play"),
    ("off_turnovers", "def_turnovers_forced"),
    ("off_turnover_pct", "def_turnover_pct"),
    ("off_pass_Sk%", "def_pass_Sk%"),
    ("off_rush_Y/A", "def_rush_Y/A"),
]

raw_rows = []
for off_col, def_col in raw_comparisons:
    raw_rows.append({"Feature": off_col, "Correlation": round(df[off_col].corr(df["made_playoffs"]), 3)})
    raw_rows.append({"Feature": def_col, "Correlation": round(df[def_col].corr(df["made_playoffs"]), 3)})

pd.DataFrame(raw_rows).set_index("Feature")
 

,Correlation
Feature,
score_pct_diff,0.704
pass_eff_diff,0.683
ypp_diff,0.555
turnover_margin,0.534
to_rate_diff,0.539
sack_rate_diff,0.387
rush_eff_diff,0.213


,Correlation
Feature,
off_score_pct,0.569
def_score_pct,-0.450
off_pass_ANY/A,0.563
def_pass_ANY/A,-0.452
off_yds_per_play,0.427
def_yds_per_play,-0.314
off_turnovers,-0.403
def_turnovers_forced,0.412
off_turnover_pct,-0.411


Observing the top result here: score_pct_diff we have r= +0.704 which means we have a strong positive relationship with making the playoffs. 
* df["score_pct_diff"] = df["off_score_pct"] - df["def_score_pct"]: Breaking this down we understand that now teams with a higher scoring efficiency differential, meaning their offense scores more efficiently than their defense allows, tend to make the playoffs.
* Offensive score percentage is r = 0.569 and defensive score percentage is r = -0.450. By combining them into one differential, we got 0.704, which is much better than either alone.

In [48]:
# Defensive advanced feature correlations

adv_retained = ["def_adv_Prss", "def_adv_MTkl", "def_adv_Air", "def_adv_YAC"]

adv_rows = []
for col in adv_retained:
    if col in df.columns:
        corr = df[col].corr(df["made_playoffs"])
        adv_rows.append({"Feature": col, "Correlation": round(corr, 3)})
 
display(pd.DataFrame(adv_rows).set_index("Feature"))

,Correlation
Feature,
def_adv_Prss,0.348
def_adv_MTkl,-0.220
def_adv_Air,0.342
def_adv_YAC,0.295


In [49]:
all_feature_cols = [c for c in df.columns if c not in key_cols + [target_col]]

print(f"Final dataset shape: {df.shape}")
print(f"Key columns: {key_cols}")
print(f"Target: {target_col}")
print(f"Total features: {len(all_feature_cols)}")

Final dataset shape: (256, 68)
Key columns: ['season', 'team']
Target: made_playoffs
Total features: 65


In [50]:
output_path = "modeling_ready_dataset.csv"
df.to_csv(output_path, index=False)